In [0]:
# ============================================================
# 02_raw_to_bronze_dallas
# ============================================================
# Purpose  : 
# Read Dallas CSV from Volume 
# rename  columns 
# generate surrogate inspection_id 
# add audit columns
# APPEND to bronze.dallas_raw

# Key facts about Dallas source:
#   114 columns: 10 core + 25 violation slots x 4 sub-fields Inspection Month, Inspection Year, Lat Long Location
#   No natural unique key - surrogate generated here in Bronze
#   Column 90 has a double-space typo in the header
# ============================================================


In [0]:
import uuid
from pyspark.sql import functions as F

CATALOG = "final_project_databi_group8"

DAL_PATH = "/Volumes/final_project_databi_group8/default/finalprojectgroup8volume/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260408.csv"
BRONZE_TABLE = f"{CATALOG}.bronze.dallas_raw"

BATCH_ID = str(uuid.uuid4())
RUN_ID   = "02_raw_to_bronze_dallas"
SCHEMA_V = "v1"

print(f"Catalog      : {CATALOG}")
print(f"Source file  : {DAL_PATH}")
print(f"Target table : {BRONZE_TABLE}")
print(f"Batch ID     : {BATCH_ID}")
print(f"Run ID       : {RUN_ID}")

In [0]:
files = [f.name for f in dbutils.fs.ls("/Volumes/final_project_databi_group8/default/finalprojectgroup8volume/")]
print("Files in Volume:")
for f in files:
    print(f"  {f}")

assert "Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260408.csv" in files, "ERROR: Dallas CSV not found."
print("\n Dallas CSV confirmed")

In [0]:
# Read raw CSV
df_dal_raw = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("multiLine",   "true")
    .option("escape",      '"')
    .option("encoding",    "UTF-8")
    .csv(DAL_PATH)
)

print(f"Row count : {df_dal_raw.count():,}")
print(f"Col count : {len(df_dal_raw.columns)}")

In [0]:
# Fix known source header typo
# Column 90 in the source file has a double space:
# 'Violation  Memo - 20' (TWO spaces between Violation and Memo) This must be corrected BEFORE the rename loop, otherwise the
# loop skips it and the column retains the double-space name, which Delta Lake will either reject or silently mangle.

bad_col  = "Violation  Memo - 20"   # two spaces - exact as in source
good_col = "Violation Memo - 20"    # one space - what the loop expects

if bad_col in df_dal_raw.columns:
    df_dal_raw = df_dal_raw.withColumnRenamed(bad_col, good_col)
    print(f"fixed : '{bad_col}' '{good_col}'")
else:
    print("Column already clean")

print(f"Total columns: {len(df_dal_raw.columns)}")


In [0]:
# Rename core columns to snake_case
df_dal = (
    df_dal_raw
    .withColumnRenamed("Restaurant Name",   "restaurant_name")
    .withColumnRenamed("Inspection Type",   "inspection_type")
    .withColumnRenamed("Inspection Date",   "inspection_date")
    .withColumnRenamed("Inspection Score",  "inspection_score")
    .withColumnRenamed("Street Number",     "street_number")
    .withColumnRenamed("Street Name",       "street_name")
    .withColumnRenamed("Street Direction",  "street_direction")
    .withColumnRenamed("Street Type",       "street_type")
    .withColumnRenamed("Street Unit",       "street_unit")
    .withColumnRenamed("Street Address",    "street_address")
    .withColumnRenamed("Zip Code",          "zip_code")
    .withColumnRenamed("Inspection Month",  "inspection_month")
    .withColumnRenamed("Inspection Year",   "inspection_year")
    .withColumnRenamed("Lat Long Location", "location_raw")
)

# Rename all 25 violation slots (100 columns)
# Original names have spaces and dashes

for i in range(1, 26):
    df_dal = (
        df_dal
        .withColumnRenamed(f"Violation Description - {i}", f"viol_desc_{i}")
        .withColumnRenamed(f"Violation Points - {i}",      f"viol_points_{i}")
        .withColumnRenamed(f"Violation Detail - {i}",      f"viol_detail_{i}")
        .withColumnRenamed(f"Violation Memo - {i}",        f"viol_memo_{i}")
    )

print(f"Total columns after rename: {len(df_dal.columns)}")


In [0]:
#  Generate surrogate inspection_id
# Dallas has NO natural unique key.
# Problem: 375 rows share the same restaurant_name + inspection_date + inspection_score (e.g. two 7-Eleven branches inspected same
# day with same score). Adding street_address as 4th component resolves all collisions.
# SHA-256 produces a stable 64-char hex string.

df_dal = df_dal.withColumn(
    "inspection_id",
    F.sha2(
        F.concat_ws("||",
            F.coalesce(F.col("restaurant_name"),  F.lit("")),
            F.coalesce(F.col("inspection_date"),  F.lit("")),
            F.coalesce(F.col("inspection_score"), F.lit("")),
            F.coalesce(F.col("street_address"),   F.lit(""))
        ), 256
    )
)

total  = df_dal.count()
unique = df_dal.select("inspection_id").distinct().count()
print(f"Total rows          : {total:,}")
print(f"Unique inspection_id: {unique:,}")
print(f"Collisions remaining: {total - unique}")


In [0]:
# Add audit metadata columns
df_dal = (
    df_dal
    .withColumn("city_source",     F.lit("DAL"))
    .withColumn("_batch_id",       F.lit(BATCH_ID))
    .withColumn("_extract_ts",     F.current_timestamp())
    .withColumn("_extract_date",   F.current_date().cast("string"))
    .withColumn("_source_file",    F.lit("Dallas_Inspections_20260408.csv"))
    .withColumn("_run_id",         F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_V))
    .withColumn("_row_hash",       F.sha2(
        F.concat_ws("||",
            F.coalesce(F.col("restaurant_name"),  F.lit("")),
            F.coalesce(F.col("inspection_date"),  F.lit("")),
            F.coalesce(F.col("inspection_score"), F.lit("")),
            F.coalesce(F.col("viol_desc_1"),      F.lit(""))
        ), 256))
)

bad_cols = [c for c in df_dal.columns
            if any(x in c for x in [" ", "-", "(", ")", ","])]
if bad_cols:
    raise ValueError(f"Bad column names remain: {bad_cols}")

print(f" All {len(df_dal.columns)} column names are Delta-safe")

display(df_dal.select(
    "inspection_id", "restaurant_name", "inspection_date",
    "inspection_score", "zip_code", "city_source",
    "_batch_id", "_extract_ts"
).limit(3))


In [0]:
# Write to Bronze — APPEND mode partitionBy - was missing in original Dallas notebook. Chicago had partitionBy; Dallas did not. 
(
    df_dal
    .write
    .format("delta")
    .mode("append")
    .partitionBy("city_source", "_extract_date")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

written = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == BATCH_ID)
    .count()
)
source = df_dal.count()

print(f"  Written to  : {BRONZE_TABLE}")
print(f"  Batch ID    : {BATCH_ID}")
print(f"  Source rows : {source:,}")
print(f"  Written rows: {written:,}")
print(f"  Match       : {'OK' if source == written else 'MISMATCH'}")

assert source == written, f"Mismatch: source={source}, written={written}"


In [0]:
# Batch history
print("All batches ever loaded into bronze.dallas_raw:")
display(spark.sql(f"""
    SELECT
        city_source,
        _extract_date,
        _extract_ts,
        LEFT(_batch_id, 8)  AS batch_short,
        _run_id,
        _schema_version,
        COUNT(*)            AS row_count
    FROM {BRONZE_TABLE}
    GROUP BY city_source, _extract_date, _extract_ts,
             _batch_id, _run_id, _schema_version
    ORDER BY _extract_ts DESC
"""))

display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))
